# Part 1 — Build a GPT-style language model

In this notebook you will build the model that you will train in Part 2. The
model reads a window of token IDs and predicts the next token at every
position.

You will:

1. implement causal self-attention,
2. combine several attention heads into multi-head attention,
3. build a Transformer block,
4. put the blocks together into a GPT model,
5. compute the next-token prediction loss, and
6. generate text with the model.

The figure below summarizes the model. `B` is the batch size and `T` is the
number of tokens in each window.

![](assets/gpt_architecture.png)

If a part of the model is unclear, the lecture slides and
[The Illustrated Transformer](https://jalammar.github.io/illustrated-transformer/)
(in particular the sections from "Self-Attention in Detail" to "The Beast With
Many Heads") explain it with more figures. Reading them is optional.

In [1]:
%load_ext autoreload
%autoreload 2

import math
import os
import torch
from tokenizers import Tokenizer

import assignment3 as a3

DATA_DIR = "./data"
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("Device:", device)

torch.manual_seed(0)
tokenizer = Tokenizer.from_file(os.path.join(DATA_DIR, "tokenizer.json"))
vocab_size = tokenizer.get_vocab_size()
print("Vocabulary size:", vocab_size)

Device: cuda
Vocabulary size: 4096


## 1. Causal self-attention

**Self-attention** lets every token collect information from the tokens before
it. An **attention head** first maps every token to three vectors with three
linear layers:

- the **query** `q`: what this token is looking for,
- the **key** `k`: what this token contains, to be matched with the queries,
- the **value** `v`: the information that this token passes on.

![](assets/attention_qkv.png)

Then, as in the figure below:

1. The **score** of token `t` for token `s` is the dot product `q[t] · k[s]`,
   so `q @ kᵀ` gives all scores at once. The scores are divided by
   $\sqrt{d_{head}}$ so that they do not grow with the head dimension.
2. The **mask** sets the scores of the later tokens to $-\infty$, because the
   model must not see the tokens that it is asked to predict. This is why it is
   called **causal** self-attention.
3. A **softmax** over every row turns the scores into **weights** that sum to 1.
   Since $e^{-\infty} = 0$, the masked tokens get a weight of 0.
4. The output `z` of every token is the weighted sum of the values.

![](assets/attention_z.png)

$$z = \text{softmax}\left(\text{mask}\left(\frac{q k^\top}{\sqrt{d_{head}}}\right)\right) v$$

**Your task here:**

Complete `AttentionHead.forward()` in [`assignment3/model.py`](assignment3/model.py)
(1 mark). The mask is provided between your two parts of the code. All tensors have a
batch dimension in front, e.g. `q` has shape `(batch_size, T, head_dim)`.

1. Compute `q`, `k`, and `v` with `self.query`, `self.key`, and `self.value`.
2. Compute `scores`, of shape `(batch_size, T, T)`. For `kᵀ`, swap the last two
   dimensions of `k` with `k.transpose(-2, -1)` or with `einops.rearrange()`.
3. Compute `weights` with a softmax of `masked_scores` over the last dimension,
   and then `z`.

Useful documentation:
- [`Tensor.transpose`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.transpose.html)
- [`einops.rearrange`](https://einops.rocks/api/rearrange/)
- [`torch.nn.functional.softmax`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.softmax.html)

In [2]:
head = a3.AttentionHead(d_model=16, head_dim=8)
x = torch.randn(2, 5, 16)  # (batch_size, T, d_model)
z = head(x)
print("z:", z.shape)  # expected output: torch.Size([2, 5, 8])
assert z.shape == (2, 5, 8), "z should have shape (batch_size, T, head_dim)"

# Every row should sum to 1, and the weights above the diagonal should be 0.
weights = head.attention_weights
print("\nAttention weights of the first window:")
print(weights[0].round(decimals=2))
assert torch.allclose(weights.sum(dim=-1), torch.ones(2, 5)), "every row of the weights should sum to 1"
assert torch.all(torch.triu(weights, diagonal=1) == 0), "the weights above the diagonal should be 0"

z: torch.Size([2, 5, 8])

Attention weights of the first window:
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4900, 0.5100, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.3200, 0.4300, 0.0000, 0.0000],
        [0.2800, 0.2600, 0.2400, 0.2200, 0.0000],
        [0.2500, 0.2400, 0.1500, 0.2000, 0.1500]])


## 2. Multi-head attention

**Multi-head attention** runs `n_heads` heads on the same input `x`, each with
its own query, key, and value layers and `head_dim = d_model // n_heads`. Their
outputs, each of shape `(batch_size, T, head_dim)`, are concatenated along the
last dimension into shape `(batch_size, T, d_model)`, and a linear layer
(`self.projection`) mixes them.

**Your task here:**

Complete `MultiHeadAttention.forward()` in [`assignment3/model.py`](assignment3/model.py)
(1 mark).

The heads are stored in `self.heads`. Run every head on
`x`, concatenate their outputs along the last dimension, and apply `self.projection`.

Useful documentation:
- [`torch.cat`](https://docs.pytorch.org/docs/stable/generated/torch.cat.html)

In [3]:
attention = a3.MultiHeadAttention(d_model=16, n_heads=4)
out = attention(x)
print("out:", out.shape)  # expected output: torch.Size([2, 5, 16])
assert out.shape == (2, 5, 16), "out should have shape (batch_size, T, d_model)"

out: torch.Size([2, 5, 16])


## 3. Transformer block

A Transformer block (red box below) has two parts: multi-head attention, which
mixes information between tokens, and an **MLP**, which processes every token
on its own. Each part takes the LayerNorm of its input, and its output is added
back to its input through a **residual connection**.

![](assets/gpt_architecture_block.png)

**Your task here:**

Complete `Block.forward()` in [`assignment3/model.py`](assignment3/model.py)
following the figure (1 mark). The layers are created in `__init__()`.

In [4]:
block = a3.Block(d_model=16, n_heads=4)
out = block(x)
print("out:", out.shape)  # expected output: torch.Size([2, 5, 16])
assert out.shape == (2, 5, 16), "out should have shape (batch_size, T, d_model)"

out: torch.Size([2, 5, 16])


## 4. The GPT model

The GPT model (red box below) is the whole model, which takes the token IDs and
returns the logits. The position embedding takes the positions `0, 1, ..., T - 1`
as input. For example, for "Once upon a time" (`T = 4`):

```
tokens:     Once   Ġupon   Ġa    Ġtime
ids:        430    438     259   398     → token_embedding(ids)          (B, T, d_model)
positions:  0      1       2     3       → position_embedding(positions) (T, d_model)
```

The two embeddings are added; the position embedding is broadcast over the batch.

![](assets/gpt_architecture_model.png)

**Your task here:**

Complete `GPT.forward()` in [`assignment3/model.py`](assignment3/model.py)
following the figure (1 mark). The layers are created in `__init__()`. Note that:
- `T` is the length of the input, which can be shorter than `context_length`
  when generating text, so create the positions from `T`.
- The positions must be on the same device as `ids`.

Useful documentation:
- [`torch.arange`](https://docs.pytorch.org/docs/stable/generated/torch.arange.html)

In [5]:
# a small model for testing; Part 2 trains a larger one
model = a3.GPT(vocab_size, d_model=64, n_layers=2, n_heads=4).to(device)
print(f"Parameters of the small model: {sum(p.numel() for p in model.parameters()):,}")
print(f"Parameters of the default model: {sum(p.numel() for p in a3.GPT(vocab_size).parameters()):,}")

val_loader = a3.get_loader(DATA_DIR, "val", batch_size=8)
x, y = next(iter(val_loader))
x, y = x.to(device), y.to(device)
logits = model(x)
print("x:", x.shape)  # expected output: torch.Size([8, 256])
print("logits:", logits.shape)  # expected output: torch.Size([8, 256, 4096])
assert logits.shape == (x.shape[0], x.shape[1], vocab_size), "logits should have shape (batch_size, T, vocab_size)"

Parameters of the small model: 644,864
Parameters of the default model: 5,326,336
x: torch.Size([8, 256])
logits: torch.Size([8, 256, 4096])


## 5. Next-token prediction loss

Predicting the next token is a classification with `vocab_size` classes at
every position, so we use the cross-entropy loss between the logits and the
targets `y`, averaged over all positions of all windows in the batch.

**Your task here:**

Complete `compute_loss()` in [`assignment3/training.py`](assignment3/training.py)
(1 mark). Note that `F.cross_entropy()` expects logits of shape `(N, num_classes)` and
targets of shape `(N,)`, so every position of every window should be one of the
`N` samples.

Before training, the model gives all tokens about the same probability, so the
loss should be close to $\ln(4096) \approx 8.3$.

Useful documentation:
- [`torch.nn.functional.cross_entropy`](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)
- [`Tensor.reshape`](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.reshape.html)

In [6]:
loss = a3.compute_loss(logits, y)
print(f"Initial loss: {loss.item():.2f}")
print(f"ln(vocab_size): {math.log(vocab_size):.2f}")
assert loss.ndim == 0, "the loss should be a scalar"

Initial loss: 8.53
ln(vocab_size): 8.32


## 6. Generate text

`GPT.generate()` in [`assignment3/model.py`](assignment3/model.py) is provided. It is used at
test time to generate text. It predicts the next token, appends it to the
input, and repeats this until it has generated `num_new_tokens` new tokens. To
generate one token, it:

1. **Computes the logits.** It gives the last `context_length` tokens of `ids`
   (the model cannot read more) to the model, and takes the raw logits at the
   last position, of shape `(B, 4096)`. The logits are divided by the
   **temperature**: a temperature below 1 makes the differences between the
   logits larger, so the likely tokens become even more likely (safer but more
   repetitive text); a temperature above 1 makes them smaller, so the
   probabilities become more even (more varied but less coherent text).
2. **Keeps the top-k tokens.** It keeps the `top_k` largest logits and sets the
   others to $-\infty$. Otherwise, the next token could be one of the thousands
   of unlikely tokens in the long tail, which would make the text random.
3. **Samples the next token.** A softmax over the remaining logits gives the
   probabilities of the top-k tokens; the removed tokens get a probability
   of 0. The next token is then **sampled** randomly according to these
   probabilities, of shape `(B, 1)`. This differs from classification, where we
   simply take the class with the highest score: sampling lets the model
   generate different stories from the same prompt.
4. **Appends the new token** to `ids`, which becomes `(B, T + 1)`.

We suggest that you read through the code of `generate()` and match its lines
with the four steps above.

The model is not trained yet, so the cell below generates random tokens. You
will train the model and try different settings in Part 2.

In [7]:
prompt = "Once upon a time"
ids = torch.tensor([tokenizer.encode(prompt).ids], device=device)
out = model.generate(ids, num_new_tokens=20, temperature=1.0, top_k=50)
print("out:", out.shape)  # expected output: torch.Size([1, 24])
print(tokenizer.decode(out[0].tolist()))

# a model with context_length = 8 should still generate more than 8 tokens
tiny_model = a3.GPT(vocab_size, context_length=8, d_model=16, n_layers=1, n_heads=2).to(device)
out = tiny_model.generate(ids, num_new_tokens=20)
print("\nout:", out.shape)  # expected output: torch.Size([1, 24])

out: torch.Size([1, 24])
Once upon a timepa mean zipper collGive entkin yesllaions scared dreams upon shine0, Just who having�

out: torch.Size([1, 24])
